# TAREFA 6
Desenvolvimento Rápido de Aplicações em Python

> Marcos Alcino Ribeiro Cussioli

> 202402394612

In [ ]:
import sqlite3
import sys
import traceback
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- CONFIG ----------
DB_FILENAME = "escola.db"  # padrão: arquivo no diretório do Colab (/content)

# ---------- Banco e utilitários ----------
_conn = None  # conexão global

def get_connection(db_filename=DB_FILENAME):
    global _conn
    if _conn is None:
        _conn = sqlite3.connect(db_filename, check_same_thread=False)
        _conn.row_factory = sqlite3.Row
        _conn.execute("PRAGMA foreign_keys = ON;")
    return _conn

def close_connection():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception as e:
        print("Erro ao fechar conexão:", e)
    finally:
        _conn = None

def create_tabelas(conn):
    c = conn.cursor()
    c.execute("""
    CREATE TABLE IF NOT EXISTS alunos (
        matricula TEXT PRIMARY KEY,
        nome TEXT NOT NULL
    );
    """)
    c.execute("""
    CREATE TABLE IF NOT EXISTS notas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        matricula_aluno TEXT NOT NULL,
        disciplina TEXT NOT NULL,
        nota REAL NOT NULL,
        FOREIGN KEY(matricula_aluno) REFERENCES alunos(matricula) ON DELETE CASCADE
    );
    """)
    conn.commit()

def row_to_dict(row):
    if row is None:
        return None
    return {k: row[k] for k in row.keys()}

def aluno_existe(conn, matricula):
    c = conn.cursor()
    c.execute("SELECT 1 FROM alunos WHERE matricula = ? LIMIT 1;", (matricula,))
    return c.fetchone() is not None

# ---------- Operações (módulos) ----------
def cadastrar_aluno(matricula, nome):
    conn = get_connection()
    aluno = {'matricula': matricula.strip(), 'nome': nome.strip()}
    if not aluno['matricula'] or not aluno['nome']:
        raise ValueError("Matrícula e nome não podem ser vazios.")
    try:
        c = conn.cursor()
        c.execute("INSERT INTO alunos (matricula, nome) VALUES (?, ?);",
                  (aluno['matricula'], aluno['nome']))
        conn.commit()
        return aluno
    except sqlite3.IntegrityError as e:
        raise sqlite3.IntegrityError(f"Matrícula '{aluno['matricula']}' já existe.") from e

def adicionar_nota(matricula, disciplina, nota_str):
    conn = get_connection()
    matricula = matricula.strip()
    disciplina = disciplina.strip()
    if not aluno_existe(conn, matricula):
        raise LookupError(f"Aluno com matrícula '{matricula}' não encontrado.")
    try:
        nota = float(nota_str)
    except ValueError as e:
        raise ValueError("Nota inválida. Use número, ex: 8.5") from e
    c = conn.cursor()
    c.execute("INSERT INTO notas (matricula_aluno, disciplina, nota) VALUES (?, ?, ?);",
              (matricula, disciplina, nota))
    conn.commit()
    return {'matricula_aluno': matricula, 'disciplina': disciplina, 'nota': nota}

def calcular_media(matricula):
    conn = get_connection()
    if not aluno_existe(conn, matricula):
        raise LookupError(f"Aluno com matrícula '{matricula}' não encontrado.")
    c = conn.cursor()
    c.execute("SELECT AVG(nota) AS media, COUNT(*) AS qtd FROM notas WHERE matricula_aluno = ?;", (matricula,))
    row = c.fetchone()
    return {'media': row['media'], 'qtd': row['qtd']}

def listar_alunos_notas_df():
    conn = get_connection()
    c = conn.cursor()
    c.execute("""
    SELECT a.matricula, a.nome, n.disciplina, n.nota
    FROM alunos a
    LEFT JOIN notas n ON a.matricula = n.matricula_aluno
    ORDER BY a.nome, n.disciplina;
    """)
    rows = c.fetchall()
    if not rows:
        return pd.DataFrame(columns=['matricula','nome','disciplina','nota'])
    data = [dict(r) for r in rows]
    df = pd.DataFrame(data)
    return df

def listar_alunos_agregado():
    """
    Retorna uma lista de dicionários: {matricula, nome, notas: [(disciplina, nota)...]}
    """
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT matricula, nome FROM alunos ORDER BY nome;")
    alunos = c.fetchall()
    resultado = []
    for a in alunos:
        matricula = a['matricula']
        nome = a['nome']
        c.execute("SELECT disciplina, nota FROM notas WHERE matricula_aluno = ? ORDER BY disciplina;", (matricula,))
        notas = [(nr['disciplina'], nr['nota']) for nr in c.fetchall()]
        resultado.append({'matricula': matricula, 'nome': nome, 'notas': notas})
    return resultado

# ---------- Interface com ipywidgets ----------
# Widgets e área de saída
out = widgets.Output(layout={'border': '1px solid gray'})
tab = widgets.Tab()

# ---- Aba: Cadastrar Aluno ----
matricula_input = widgets.Text(description="Matrícula:", placeholder="Ex: 123")
nome_input = widgets.Text(description="Nome:", placeholder="Ex: João")
botao_cadastrar = widgets.Button(description="Cadastrar", button_style='success')
label_result_cadastrar = widgets.HTML()

def on_cadastrar_click(b):
    with out:
        clear_output(wait=True)
        try:
            aluno = cadastrar_aluno(matricula_input.value, nome_input.value)
            print(f"Aluno cadastrado: {aluno['matricula']} - {aluno['nome']}")
            matricula_input.value = ""
            nome_input.value = ""
        except Exception as e:
            print("Erro:", e)

botao_cadastrar.on_click(on_cadastrar_click)
aba_cadastrar = widgets.VBox([matricula_input, nome_input, botao_cadastrar])

# ---- Aba: Adicionar Nota ----
matricula_nota_input = widgets.Text(description="Matrícula:", placeholder="Ex: 123")
disciplina_input = widgets.Text(description="Disciplina:", placeholder="Ex: Matemática")
nota_input = widgets.Text(description="Nota:", placeholder="Ex: 8.5")
botao_adicionar_nota = widgets.Button(description="Adicionar Nota", button_style='info')

def on_adicionar_nota_click(b):
    with out:
        clear_output(wait=True)
        try:
            resultado = adicionar_nota(matricula_nota_input.value, disciplina_input.value, nota_input.value)
            print(f"Nota adicionada: {resultado['matricula_aluno']} - {resultado['disciplina']} - {resultado['nota']}")
            disciplina_input.value = ""
            nota_input.value = ""
        except Exception as e:
            print("Erro:", e)

botao_adicionar_nota.on_click(on_adicionar_nota_click)
aba_adicionar_nota = widgets.VBox([matricula_nota_input, disciplina_input, nota_input, botao_adicionar_nota])

# ---- Aba: Calcular Média ----
matricula_media_input = widgets.Text(description="Matrícula:", placeholder="Ex: 123")
botao_calcular_media = widgets.Button(description="Calcular Média", button_style='warning')

def on_calcular_media_click(b):
    with out:
        clear_output(wait=True)
        try:
            res = calcular_media(matricula_media_input.value)
            if res['qtd'] == 0 or res['media'] is None:
                print(f"O aluno {matricula_media_input.value} não tem notas registradas.")
            else:
                print(f"Média do aluno {matricula_media_input.value}: {res['media']:.2f} (baseado em {res['qtd']} nota(s))")
        except Exception as e:
            print("Erro:", e)

botao_calcular_media.on_click(on_calcular_media_click)
aba_calcular_media = widgets.VBox([matricula_media_input, botao_calcular_media])

# ---- Aba: Listar ----
botao_listar = widgets.Button(description="Atualizar Lista", button_style='')
botao_exibir_join = widgets.Button(description="Exibir (JOIN)", button_style='')
botao_exibir_agregado = widgets.Button(description="Exibir (Agrupado)", button_style='')

def on_listar_click(b):
    with out:
        clear_output(wait=True)
        try:
            df = listar_alunos_notas_df()
            if df.empty:
                print("Nenhum aluno/notas cadastrados.")
            else:
                display(df)  # mostra DataFrame com colunas: matricula, nome, disciplina, nota
        except Exception as e:
            print("Erro ao listar:", e)

def on_exibir_agregado_click(b):
    with out:
        clear_output(wait=True)
        try:
            ag = listar_alunos_agregado()
            if not ag:
                print("Nenhum aluno cadastrado.")
            else:
                for a in ag:
                    print(f"Matrícula: {a['matricula']} - Nome: {a['nome']} - Notas: {a['notas']}")
        except Exception as e:
            print("Erro:", e)

botao_listar.on_click(on_listar_click)
botao_exibir_join.on_click(on_listar_click)
botao_exibir_agregado.on_click(on_exibir_agregado_click)
aba_listar = widgets.VBox([widgets.HBox([botao_listar, botao_exibir_join, botao_exibir_agregado])])

# ---- Aba: Config / Sair ----
info_db = widgets.HTML(value=f"<b>Arquivo de banco atual:</b> <code>{DB_FILENAME}</code>")
botao_montar_drive = widgets.Button(description="Montar Google Drive (opcional)", button_style='')
db_path_input = widgets.Text(value=DB_FILENAME, description="DB Path:")
botao_mudar_db = widgets.Button(description="Mudar DB e reconectar", button_style='info')
botao_fechar = widgets.Button(description="Sair e Fechar DB", button_style='danger')

def on_montar_drive_click(b):
    with out:
        clear_output(wait=True)
        try:
            # tenta montar; só funciona no Colab
            from google.colab import drive
            drive.mount('/content/drive')
            print("Drive montado em /content/drive. Agora atualize o caminho do DB se quiser salvar lá.")
        except Exception as e:
            print("Não foi possível montar o Drive (talvez não esteja em Colab ou ocorreu erro):", e)

def on_mudar_db_click(b):
    global DB_FILENAME
    with out:
        clear_output(wait=True)
        try:
            newpath = db_path_input.value.strip()
            if not newpath:
                print("Forneça um caminho válido para o DB.")
                return
            # fecha a conexão atual, atualiza nome e reabre
            close_connection()
            DB_FILENAME = newpath
            conn = get_connection(DB_FILENAME)
            create_tabelas(conn)
            info_db.value = f"<b>Arquivo de banco atual:</b> <code>{DB_FILENAME}</code>"
            print("Conectado ao DB:", DB_FILENAME)
        except Exception as e:
            print("Erro ao mudar DB:", e)
            traceback.print_exc()

def disable_all_widgets():
    # desativa controles após sair
    widget_list = [matricula_input, nome_input, botao_cadastrar,
                   matricula_nota_input, disciplina_input, nota_input, botao_adicionar_nota,
                   matricula_media_input, botao_calcular_media,
                   botao_listar, botao_exibir_join, botao_exibir_agregado,
                   botao_montar_drive, db_path_input, botao_mudar_db, botao_fechar]
    for w in widget_list:
        try:
            w.disabled = True
        except:
            pass

def on_fechar_click(b):
    with out:
        clear_output(wait=True)
        try:
            close_connection()
            print("Conexão com o banco fechada. A interface será desativada.")
        except Exception as e:
            print("Erro ao fechar conexão:", e)
        disable_all_widgets()

botao_montar_drive.on_click(on_montar_drive_click)
botao_mudar_db.on_click(on_mudar_db_click)
botao_fechar.on_click(on_fechar_click)

aba_config = widgets.VBox([info_db,
                           widgets.HBox([db_path_input, botao_mudar_db]),
                           widgets.HTML("<i>Se quiser persistir o arquivo entre sessões, monte o Google Drive e mude o caminho do DB para algo em /content/drive/MyDrive/...</i>"),
                           botao_montar_drive,
                           widgets.HTML("<hr>"),
                           botao_fechar])

# ---- Monta as abas ----
tab_children = [aba_cadastrar, aba_adicionar_nota, aba_calcular_media, aba_listar, aba_config]
tab.children = tab_children
tab.set_title(0, "Cadastrar")
tab.set_title(1, "Adicionar Nota")
tab.set_title(2, "Calcular Média")
tab.set_title(3, "Listar")
tab.set_title(4, "Config / Sair")

# Inicializa DB e exibe interface
try:
    conn = get_connection(DB_FILENAME)
    create_tabelas(conn)
    display(tab)
    display(out)
    with out:
        print("Interface pronta. Use as abas para operar. DB:", DB_FILENAME)
except Exception as e:
    print("Erro ao inicializar a interface:", e)
    traceback.print_exc()


# MONTANDO O DB:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DB_FILENAME = "/content/drive/MyDrive/escola.db"
